In [12]:
import pandas as pd # type: ignore
import numpy as np # type: ignore
import os
import re 
import unicodedata

df = pd.read_csv('C:\\Users\\ethan\\Downloads\\One-Fight-Analysis\\fight_records.csv')
df.head()

,Fighter,Opponent,Result,Method,Date
0,Eduard “Landslide” Folayang,A-Sol Kwon,WIN,Unanimous Decision,3-Sep-11
1,Leandro “Brodinho” Issa,Soo Chul Kim,WIN,Unanimous Decision,3-Sep-11
2,Alex “Little Rock” Silva,Geje Eustaquio,LOSS,Unanimous Decision,11-Feb-12
3,Geje “Gravity” Eustaquio,Alex Silva,WIN,Unanimous Decision,11-Feb-12
4,Honorio “The Rock” Banario,Bae Yong Kwon,LOSS,Submission,11-Feb-12


In [ ]:
#Cleaning the Data
# --- Step 1: Removing all the nicknames

# Remove nicknames (anything inside quotes) from both Fighter and Opponent
df["Fighter"] = df["Fighter"].apply(lambda x: re.sub(r'“.*?”', '', str(x)).strip())
df["Opponent"] = df["Opponent"].apply(lambda x: re.sub(r'“.*?”', '', str(x)).strip())

# --- Step 2: Define a normalization function 
def normalize_name(name):
    # force to string, remove accents, collapse spaces, strip edges
    name = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("utf-8")
    name = re.sub(r"\s+", " ", name).strip()
    return name

# --- Step 3: Apply to Fighter and Opponent columns ---
df["Fighter"] = df["Fighter"].apply(normalize_name)
df["Opponent"] = df["Opponent"].apply(normalize_name)

# --- Step 4: Rebuild the FightKey using cleaned names ---
df["FightKey"] = df.apply(
    lambda row: tuple(sorted([row["Fighter"], row["Opponent"]])) + (row["Date"],),
    axis=1
)

# --- Step 5: Drop duplicates ---
df = df.drop_duplicates(subset=["FightKey"])
df = df.drop(columns=["Unnamed: 5", "Unnamed: 6", "Unnamed: 7","FightKey"], errors="ignore")
df.head()


Fighter     3025
Opponent    3025
Result      3025
Method      3025
Date        3025
dtype: int64


,Fighter,Opponent,Result,Method,Date
0,Eduard Folayang,A-Sol Kwon,WIN,Unanimous Decision,3-Sep-11
1,Leandro Issa,Soo Chul Kim,WIN,Unanimous Decision,3-Sep-11
2,Alex Silva,Geje Eustaquio,LOSS,Unanimous Decision,11-Feb-12
4,Honorio Banario,Bae Yong Kwon,LOSS,Submission,11-Feb-12
5,Eduard Folayang,Ole Laursen,LOSS,Split Decision,31-Mar-12


In [20]:
# Initialize Elo ratings
initial_elo = 1000 
elo = {}
peak = {}
k_factor = 200 # up to 200 as fighters have matches at least 5x rarelier than in chess - for better comparison

In [21]:
# Function to calculate the expected score
def expected_score(elo_a, elo_b):
    return 1 / (1 + 10**((elo_b - elo_a) / 400))

# Function to update Elo ratings
def update_elo(winner_elo, loser_elo, k_factor,method):
    multiplier = 0.8 if method == 'Split Decision' else 1
    expected_win = expected_score(winner_elo, loser_elo)
    new_winner_elo = winner_elo + k_factor * (1 - expected_win)
    new_loser_elo = loser_elo + k_factor * (0 - (1 - expected_win))
    return round(new_winner_elo, 2), round(new_loser_elo, 2)

In [26]:

# Step 3: Iterate row by row
for row in df.itertuples(index=False):
    fighter, opponent, result,method = row.Fighter, row.Opponent, row.Result, row.Method

    # Initialize if not seen before
    if fighter not in elo:
        elo[fighter] = initial_elo
        peak[fighter] = initial_elo
    if opponent not in elo:
        elo[opponent] = initial_elo
        peak[opponent] = initial_elo

    # Current ratings
    rating_f, rating_o = elo[fighter], elo[opponent]

    # Update Elo based on fight result
    if result == "WIN":   # fighter won
        new_f, new_o = update_elo(rating_f, rating_o, 200,method)
    elif result == "LOSS":  # fighter lost
        new_o, new_f = update_elo(rating_o, rating_f, 200,method)
    else:  # draws / other cases = no change
        new_f, new_o = rating_f, rating_o

    # Save updated ratings
    elo[fighter], elo[opponent] = new_f, new_o

    # Update peak ratings
    peak[fighter] = max(peak[fighter], new_f)
    peak[opponent] = max(peak[opponent], new_o)


In [24]:
results = pd.DataFrame({
    "Fighter": list(elo.keys()),
    "FinalElo": list(elo.values()),
    "PeakElo": [peak[f] for f in elo.keys()]
})

# Sort by Final Elo (highest first)
results = results.sort_values(by="FinalElo", ascending=False).reset_index(drop=True)
results.head()


,Fighter,FinalElo,PeakElo
0,Nabil Anane,1715.78,1715.78
1,Xiong Jing Nan,1606.98,1606.98
2,Demetrious Johnson,1603.55,1603.55
3,Superlek Kiatmoo9,1590.43,1733.82
4,Jaosuayai Mor Krungthepthonburi,1563.02,1563.02


In [27]:
# Save to CSV
results.to_csv("elo_rankings.csv", index=False)


In [7]:
athletestat = pd.read_csv(
    r'C:\Users\ethan\Downloads\One-Fight-Analysis\athlete_stats.csv',
    encoding="utf-8"
)
results = pd.read_csv(
    r'C:\Users\ethan\Downloads\One-Fight-Analysis\elo_rankings.csv',
    encoding="utf-8"
)

athletestat.head()


,Name,Weight (kg),Height (cm),Country,Age,Team,Sport,Gender,Weight ClassWIN,WIN,...,W Submission,W Disqualification,W Unanimous Decision,W Split Decision,L TKO,L Submission,L Disqualification,L Unanimous Decision,L Split Decision,Total Fight
0,Alexis “Barboza” Nicolas,76.6,183,France,27,Mahmoudi Gym,Kickboxing,Male,Lightweight,2,...,0,0,2,0,0,0,0,1,0,4
1,Allycia Hellen Rodrigues,52.2,161,Brazil,27,Phuket Fight Club,Muay Thai,Female,Strawweight,5,...,0,0,2,0,1,0,0,0,0,6
2,Anatoly “Sladkiy” Malykhin,92.8,180,Russia,37,Golden Team,MMA,Male,Middleweight,6,...,0,0,0,0,0,0,0,0,1,7
3,“Unstoppable” Angela Lee,56.1,164,Singapore / United States,29,Evolve MMA / United MMA,MMA,Female,Strawweight,11,...,8,0,2,0,0,0,0,2,0,14
4,Chingiz “Chinga” Allazov,70.1,181,Azerbaijan / Belarus,31,Gridin Gym,Kickboxing,Male,Featherweight,5,...,0,0,2,0,0,0,0,0,1,6


In [8]:
# --- Step 2: Define a cleaning function ---
def clean_name(name):
    name = str(name)

    # Fix Excel encoding issues: remove nickname parts like â€œBarbozaâ€
    name = re.sub(r"â€œ.*?â€", "", name)

    # Also handle proper curly quotes in case they appear
    name = re.sub(r"[“”].*?[“”]", "", name)

    # Remove straight-quote nicknames too ("Sladkiy")
    name = re.sub(r"\".*?\"", "", name)

    # Normalize accents (René → Rene)
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("utf-8")

    # Collapse multiple spaces into one
    name = re.sub(r"\s+", " ", name).strip()

    return name

# --- Step 3: Apply cleaning to Name column ---
athletestat["Name"] = athletestat["Name"].apply(clean_name)

# --- Step 4: Inspect cleaned names ---
print(athletestat["Name"].head(20))

0               Alexis Nicolas
1     Allycia Hellen Rodrigues
2             Anatoly Malykhin
3                   Angela Lee
4              Chingiz Allazov
5                Christian Lee
6               Danielle Kelly
7           Demetrious Johnson
8             Denice Zamboanga
9             Fabricio Andrade
10               Jackie Buntan
11               Jarred Brooks
12           Jonathan Di Bella
13           Jonathan Haggerty
14               Joseph Lasiri
15                Joshua Pacio
16                Kade Ruotolo
17               Masaaki Noiri
18               Mayssa Bastos
19                 Nabil Anane
Name: Name, dtype: object


In [ ]:
athletestat.to_csv(
    r'C:\Users\ethan\Downloads\One-Fight-Analysis\athlete_stats_clean.csv',
    index=False
)

In [9]:
merged = athletestat.merge(
    results,
    how="inner",
    left_on="Name",
    right_on="Fighter"
)
merged = merged.drop(columns=["Fighter"])
merged.head()

,Name,Weight (kg),Height (cm),Country,Age,Team,Sport,Gender,Weight ClassWIN,WIN,...,W Unanimous Decision,W Split Decision,L TKO,L Submission,L Disqualification,L Unanimous Decision,L Split Decision,Total Fight,FinalElo,PeakElo
0,Alexis Nicolas,76.6,183,France,27,Mahmoudi Gym,Kickboxing,Male,Lightweight,2,...,2,0,0,0,0,1,0,4,1134.40,1257.16
1,Allycia Hellen Rodrigues,52.2,161,Brazil,27,Phuket Fight Club,Muay Thai,Female,Strawweight,5,...,2,0,1,0,0,0,0,6,1322.46,1322.46
2,Anatoly Malykhin,92.8,180,Russia,37,Golden Team,MMA,Male,Middleweight,6,...,0,0,0,0,0,0,1,7,1356.83,1493.64
3,Angela Lee,56.1,164,Singapore / United States,29,Evolve MMA / United MMA,MMA,Female,Strawweight,11,...,2,0,0,0,0,2,0,14,1343.12,1429.72
4,Chingiz Allazov,70.1,181,Azerbaijan / Belarus,31,Gridin Gym,Kickboxing,Male,Featherweight,5,...,2,0,0,0,0,0,1,6,1454.70,1454.70


In [11]:
merged.to_csv(
    r'C:\Users\ethan\Downloads\One-Fight-Analysis\merged_clean.csv',
    index=False
)